# 🚀 GTM Business Intelligence Platform
### AI-Powered Go-To-Market Analytics | Enterprise E-Commerce Dataset (100K+ Orders)

This notebook extends the ERP simulation with a **GTM BI layer** that mirrors the analytics workflow of a Business Intelligence team supporting Go-To-Market leadership:

- 📊 **GTM KPI Dashboard** — Revenue growth, customer retention, seller productivity, regional concentration
- 🤖 **LLM Insight Layer** — Claude API generates executive-ready insights from live data
- 💬 **Natural Language → SQL Engine** — Ask questions in plain English, get SQL + results

> **Tools used:** Python, pandas, MySQL, Claude API (Anthropic), matplotlib, Streamlit (see `streamlit_app.py`)

---
## ⚙️ Cell 1 — Setup & Dependencies

In [ ]:
# Install required packages (run once)
# !pip install anthropic mysql-connector-python pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import mysql.connector
import anthropic
import json
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#3a3d4d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
ACCENT   = '#7c6af7'   # purple
ACCENT2  = '#22d3ee'   # cyan
ACCENT3  = '#f97316'   # orange
POSITIVE = '#4ade80'   # green
NEGATIVE = '#f87171'   # red

print('✅ Libraries loaded successfully')

---
## 🔌 Cell 2 — Database & API Connection

In [ ]:
# ── MySQL Connection ─────────────────────────────────────────
# Update credentials to match your local MySQL setup
DB_CONFIG = {
    'host':     'localhost',
    'user':     'root',
    'password': 'your_password',   # ← update this
    'database': 'ecommerce'
}

def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

def run_query(sql, params=None):
    """Run a SQL query and return a pandas DataFrame."""
    conn = get_connection()
    df = pd.read_sql(sql, conn, params=params)
    conn.close()
    return df

# ── Anthropic (Claude) API ───────────────────────────────────
# Get your free API key at: https://console.anthropic.com
# Store it safely — never commit keys to GitHub!
import os
ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'your_api_key_here')  # ← use env var
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def ask_claude(prompt, system=None):
    """Send a prompt to Claude and return the text response."""
    messages = [{'role': 'user', 'content': prompt}]
    kwargs = {'model': 'claude-haiku-4-5-20251001', 'max_tokens': 1024, 'messages': messages}
    if system:
        kwargs['system'] = system
    response = claude_client.messages.create(**kwargs)
    return response.content[0].text

# ── Test connections ─────────────────────────────────────────
try:
    test = run_query('SELECT COUNT(*) as total_orders FROM orders')
    print(f'✅ MySQL connected — {test.iloc[0,0]:,} orders found')
except Exception as e:
    print(f'⚠️  MySQL connection failed: {e}')
    print('   → Falling back to CSV demo mode for LLM/NL→SQL cells')

print('✅ Claude API client initialised')

---
## 📊 MODULE 1 — GTM Revenue KPIs
> *Mirrors GTM BI team focus: revenue growth, trends, category performance*

In [ ]:
# ── KPI 1.1  Monthly Revenue & MoM Growth Rate ───────────────
sql_monthly_revenue = """
    SELECT
        DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS month,
        ROUND(SUM(oi.price + oi.freight_value), 2)       AS total_revenue,
        COUNT(DISTINCT o.order_id)                        AS total_orders
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp BETWEEN '2017-01-01' AND '2018-12-31'
    GROUP BY month
    ORDER BY month
"""
df_rev = run_query(sql_monthly_revenue)
df_rev['mom_growth_pct'] = df_rev['total_revenue'].pct_change() * 100
df_rev['cum_revenue']    = df_rev['total_revenue'].cumsum()

# Summary KPIs
total_rev_2017 = df_rev[df_rev['month'].str.startswith('2017')]['total_revenue'].sum()
total_rev_2018 = df_rev[df_rev['month'].str.startswith('2018')]['total_revenue'].sum()
yoy_growth     = ((total_rev_2018 - total_rev_2017) / total_rev_2017) * 100
avg_mom_growth = df_rev['mom_growth_pct'].mean()

print('━'*55)
print('  📌 GTM KPI 1 — Revenue Performance')
print('━'*55)
print(f'  Total Revenue 2017 : R$ {total_rev_2017:>12,.2f}')
print(f'  Total Revenue 2018 : R$ {total_rev_2018:>12,.2f}')
print(f'  YoY Revenue Growth : {yoy_growth:>+.1f}%')
print(f'  Avg MoM Growth     : {avg_mom_growth:>+.1f}%')
print('━'*55)

df_rev.tail(6)[['month','total_revenue','mom_growth_pct']].round(2)

In [ ]:
# ── Chart 1 — Revenue trend + MoM growth ────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
fig.patch.set_facecolor('#0f1117')

# Revenue line
ax1.plot(df_rev['month'], df_rev['total_revenue']/1e6, color=ACCENT, linewidth=2.5, marker='o', markersize=4)
ax1.fill_between(df_rev['month'], df_rev['total_revenue']/1e6, alpha=0.15, color=ACCENT)
ax1.set_title('Monthly Revenue Trend (GTM KPI — Revenue Growth)', fontsize=14, color='white', pad=12)
ax1.set_ylabel('Revenue (R$ Million)', color='#8b949e')
ax1.tick_params(axis='x', rotation=45, labelsize=8)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('R$ %.1fM'))
ax1.grid(True, alpha=0.3)

# MoM growth bars
colors = [POSITIVE if v >= 0 else NEGATIVE for v in df_rev['mom_growth_pct'].fillna(0)]
ax2.bar(df_rev['month'], df_rev['mom_growth_pct'].fillna(0), color=colors, alpha=0.85)
ax2.axhline(0, color='white', linewidth=0.8, alpha=0.4)
ax2.set_ylabel('MoM %', color='#8b949e')
ax2.tick_params(axis='x', rotation=45, labelsize=7)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('/home/claude/gtm_upgrade/chart_revenue_trend.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Chart saved.')

In [ ]:
# ── KPI 1.2  Top GTM Categories by Revenue ──────────────────
sql_categories = """
    SELECT
        COALESCE(p.product_category_name, 'Unknown') AS category,
        ROUND(SUM(oi.price + oi.freight_value), 2)   AS revenue,
        COUNT(DISTINCT oi.order_id)                   AS orders,
        ROUND(AVG(oi.price), 2)                       AS avg_selling_price
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN orders o   ON oi.order_id   = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY category
    ORDER BY revenue DESC
    LIMIT 10
"""
df_cat = run_query(sql_categories)
df_cat['revenue_share_pct'] = (df_cat['revenue'] / df_cat['revenue'].sum() * 100).round(2)

print('━'*55)
print('  📌 GTM KPI 2 — Category Revenue Performance')
print('━'*55)
print(df_cat[['category','revenue','revenue_share_pct','avg_selling_price']].to_string(index=False))

---
## 👥 MODULE 2 — GTM Customer Retention KPIs
> *Mirrors GTM BI team focus: retention rate, churn signals, repeat buyer trends*

In [ ]:
# ── KPI 2.1  Customer Retention Rate ────────────────────────
sql_retention = """
    WITH first_order AS (
        SELECT customer_unique_id,
               MIN(order_purchase_timestamp) AS first_purchase
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE order_status = 'delivered'
        GROUP BY customer_unique_id
    ),
    returning AS (
        SELECT f.customer_unique_id
        FROM first_order f
        JOIN orders o ON o.customer_id IN (
            SELECT customer_id FROM customers WHERE customer_unique_id = f.customer_unique_id
        )
        WHERE o.order_purchase_timestamp > f.first_purchase
          AND o.order_status = 'delivered'
        GROUP BY f.customer_unique_id
    )
    SELECT
        COUNT(DISTINCT f.customer_unique_id)  AS total_customers,
        COUNT(DISTINCT r.customer_unique_id)  AS returning_customers
    FROM first_order f
    LEFT JOIN returning r ON f.customer_unique_id = r.customer_unique_id
"""
df_ret = run_query(sql_retention)
retention_rate = (df_ret['returning_customers'].iloc[0] / df_ret['total_customers'].iloc[0]) * 100
churn_rate     = 100 - retention_rate

print('━'*55)
print('  📌 GTM KPI 3 — Customer Retention')
print('━'*55)
print(f'  Total Unique Customers  : {df_ret["total_customers"].iloc[0]:,}')
print(f'  Returning Customers     : {df_ret["returning_customers"].iloc[0]:,}')
print(f'  Retention Rate          : {retention_rate:.2f}%')
print(f'  Churn Rate              : {churn_rate:.2f}%')
print('━'*55)
print()
if retention_rate < 10:
    print('  ⚠️  ALERT: Retention rate critically low — high one-time buyer concentration.')
    print('  → GTM intervention: loyalty programme, re-engagement campaign recommended.')

In [ ]:
# ── KPI 2.2  New vs Returning customer trend ─────────────────
sql_new_vs_ret = """
    WITH ranked AS (
        SELECT
            c.customer_unique_id,
            o.order_purchase_timestamp,
            DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS month,
            ROW_NUMBER() OVER (
                PARTITION BY c.customer_unique_id
                ORDER BY o.order_purchase_timestamp
            ) AS order_rank
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'
    )
    SELECT
        month,
        SUM(CASE WHEN order_rank = 1 THEN 1 ELSE 0 END) AS new_customers,
        SUM(CASE WHEN order_rank > 1 THEN 1 ELSE 0 END) AS returning_customers
    FROM ranked
    WHERE month BETWEEN '2017-01' AND '2018-12'
    GROUP BY month
    ORDER BY month
"""
df_nv = run_query(sql_new_vs_ret)

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')
ax.stackplot(df_nv['month'],
             df_nv['new_customers'], df_nv['returning_customers'],
             labels=['New Customers', 'Returning Customers'],
             colors=[ACCENT, POSITIVE], alpha=0.85)
ax.set_title('New vs Returning Customers (GTM KPI — Retention Trend)', fontsize=13, color='white', pad=10)
ax.set_ylabel('Customer Count', color='#8b949e')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.legend(loc='upper left', facecolor='#1a1d27', labelcolor='white')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('/home/claude/gtm_upgrade/chart_retention_trend.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## 🏪 MODULE 3 — GTM Seller Productivity KPIs
> *Mirrors GTM BI focus: seller/rep performance, top vs bottom productivity*

In [ ]:
# ── KPI 3.1  Seller Performance Scorecard ───────────────────
sql_seller = """
    SELECT
        oi.seller_id,
        s.seller_state,
        COUNT(DISTINCT oi.order_id)                   AS total_orders,
        ROUND(SUM(oi.price + oi.freight_value), 2)    AS total_revenue,
        ROUND(AVG(oi.price), 2)                       AS avg_order_value,
        COUNT(DISTINCT oi.product_id)                 AS unique_products,
        DENSE_RANK() OVER (ORDER BY SUM(oi.price) DESC) AS revenue_rank
    FROM order_items oi
    JOIN sellers s ON oi.seller_id = s.seller_id
    JOIN orders o  ON oi.order_id  = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY oi.seller_id, s.seller_state
    ORDER BY total_revenue DESC
    LIMIT 15
"""
df_seller = run_query(sql_seller)

# Productivity tiers
p80 = df_seller['total_revenue'].quantile(0.8)
p50 = df_seller['total_revenue'].quantile(0.5)
def tier(rev):
    if rev >= p80:  return '🟢 High'
    if rev >= p50:  return '🟡 Mid'
    return '🔴 Low'
df_seller['productivity_tier'] = df_seller['total_revenue'].apply(tier)

print('━'*60)
print('  📌 GTM KPI 4 — Seller Productivity Scorecard (Top 15)')
print('━'*60)
print(df_seller[['seller_id','seller_state','total_revenue','avg_order_value','productivity_tier']]
      .head(10).to_string(index=False))

In [ ]:
# ── KPI 3.2  Regional Revenue Concentration ─────────────────
sql_regional = """
    SELECT
        c.customer_state                              AS state,
        COUNT(DISTINCT o.order_id)                    AS total_orders,
        ROUND(SUM(oi.price + oi.freight_value), 2)   AS total_revenue,
        COUNT(DISTINCT c.customer_unique_id)          AS unique_customers
    FROM orders o
    JOIN customers c  ON o.customer_id  = c.customer_id
    JOIN order_items oi ON o.order_id   = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
    ORDER BY total_revenue DESC
    LIMIT 10
"""
df_reg = run_query(sql_regional)
df_reg['revenue_share'] = (df_reg['total_revenue'] / df_reg['total_revenue'].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0f1117')
bars = ax.barh(df_reg['state'], df_reg['total_revenue']/1e6, color=ACCENT, alpha=0.85)
for bar, share in zip(bars, df_reg['revenue_share']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{share}%', va='center', color='#8b949e', fontsize=9)
ax.set_title('Regional Revenue Concentration (GTM KPI — Market Penetration)', fontsize=13, color='white', pad=10)
ax.set_xlabel('Revenue (R$ Million)', color='#8b949e')
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.2)
plt.tight_layout()
plt.savefig('/home/claude/gtm_upgrade/chart_regional.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

---
## 🤖 MODULE 4 — LLM Insight Layer (Claude API)
> *Demonstrates: AI-powered executive insights from live GTM data — core Nutanix BI requirement*

In [ ]:
# ── LLM Insight 1 — Revenue Executive Summary ────────────────

# Build a data summary string from live query results
revenue_summary = f"""
GTM Revenue Data Summary:
- Total Revenue 2017: R$ {total_rev_2017:,.0f}
- Total Revenue 2018: R$ {total_rev_2018:,.0f}
- Year-over-Year Growth: {yoy_growth:.1f}%
- Average Month-over-Month Growth: {avg_mom_growth:.1f}%
- Top Revenue Category: {df_cat.iloc[0]['category']} ({df_cat.iloc[0]['revenue_share_pct']}% of revenue)
- 2nd Category: {df_cat.iloc[1]['category']} ({df_cat.iloc[1]['revenue_share_pct']}%)
- Top Revenue State: {df_reg.iloc[0]['state']} ({df_reg.iloc[0]['revenue_share']}% of total)
"""

prompt_revenue = f"""
You are a Senior Business Intelligence Analyst preparing a briefing for the VP of Sales.
Based on the following GTM revenue data, generate a concise executive summary with:
1. 3 key insight bullets (start each with an emoji)
2. 1 risk flag
3. 1 recommended GTM action

Keep the total response under 200 words. Use business language appropriate for a C-suite audience.

{revenue_summary}
"""

print('⏳ Calling Claude API for revenue insights...\n')
insight_revenue = ask_claude(prompt_revenue)

print('━'*60)
print('  🤖 AI EXECUTIVE INSIGHT — Revenue Performance')
print('  Generated by Claude (Anthropic) from live GTM data')
print('━'*60)
print(insight_revenue)
print('━'*60)

In [ ]:
# ── LLM Insight 2 — Customer Retention Alert ────────────────

retention_summary = f"""
GTM Customer Retention Data:
- Total Unique Customers: {df_ret['total_customers'].iloc[0]:,}
- Returning Customers: {df_ret['returning_customers'].iloc[0]:,}
- Retention Rate: {retention_rate:.2f}%
- Churn Rate: {churn_rate:.2f}%
- Top customer state: {df_reg.iloc[0]['state']} with {df_reg.iloc[0]['unique_customers']:,} customers
"""

prompt_retention = f"""
You are a GTM Business Intelligence Analyst preparing a churn risk briefing for the Customer Success VP.
Based on the customer retention data below, generate:
1. A one-sentence diagnosis of the retention situation
2. 2 likely root causes
3. 2 specific GTM interventions to improve retention

Keep response under 150 words. Be direct and specific.

{retention_summary}
"""

print('⏳ Calling Claude API for retention insights...\n')
insight_retention = ask_claude(prompt_retention)

print('━'*60)
print('  🤖 AI EXECUTIVE INSIGHT — Customer Retention Alert')
print('━'*60)
print(insight_retention)
print('━'*60)

In [ ]:
# ── LLM Insight 3 — Seller Productivity Briefing ────────────

top3_sellers = df_seller.head(3)[['seller_state','total_revenue','avg_order_value']].to_string(index=False)

seller_summary = f"""
GTM Seller Productivity Data (Top 3 sellers shown):
{top3_sellers}

High-productivity sellers (top 20%): {(df_seller['productivity_tier']=='🟢 High').sum()} sellers
Mid-productivity sellers (50-80%): {(df_seller['productivity_tier']=='🟡 Mid').sum()} sellers
Low-productivity sellers (bottom 50%): {(df_seller['productivity_tier']=='🔴 Low').sum()} sellers
"""

prompt_seller = f"""
You are a GTM BI Analyst briefing the Head of Sales Operations.
Based on seller productivity data, provide:
1. A productivity tier summary (1-2 sentences)
2. One early warning signal
3. One coaching recommendation for low-tier sellers

Under 120 words. Use GTM/sales terminology.

{seller_summary}
"""

print('⏳ Calling Claude API for seller insights...\n')
insight_seller = ask_claude(prompt_seller)

print('━'*60)
print('  🤖 AI EXECUTIVE INSIGHT — Seller Productivity')
print('━'*60)
print(insight_seller)
print('━'*60)

---
## 💬 MODULE 5 — Natural Language → SQL Engine
> *Demonstrates: conversational analytics — lightweight ThoughtSpot-style NL query*

In [ ]:
# ── NL → SQL Schema Context ──────────────────────────────────
DB_SCHEMA = """
Database: ecommerce (Brazilian E-Commerce, 100K+ orders)

Tables and key columns:
- orders(order_id, customer_id, order_status, order_purchase_timestamp, order_delivered_customer_date)
- customers(customer_id, customer_unique_id, customer_city, customer_state)
- order_items(order_id, product_id, seller_id, price, freight_value)
- products(product_id, product_category_name, product_weight_g)
- sellers(seller_id, seller_city, seller_state)
- payments(order_id, payment_type, payment_installments, payment_value)
- geolocation(geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state)

Notes:
- Join orders→customers on customer_id
- Join orders→order_items on order_id
- Join order_items→products on product_id
- Join order_items→sellers on seller_id
- Use order_status = 'delivered' for revenue analysis
- Revenue = price + freight_value from order_items
"""

def nl_to_sql(user_question):
    """Convert a natural language question to SQL using Claude, then execute it."""
    
    system_prompt = f"""
You are a SQL expert. Convert the user's natural language question into a valid MySQL query.
Use ONLY the tables and columns in this schema:

{DB_SCHEMA}

Rules:
- Return ONLY the SQL query, no explanation, no markdown, no backticks
- Always add LIMIT 20 unless user specifies otherwise
- Use table aliases for readability
- For revenue, use SUM(oi.price + oi.freight_value)
"""
    
    sql_query = ask_claude(user_question, system=system_prompt).strip()
    
    print(f'❓ Question : {user_question}')
    print(f'🔧 Generated SQL:')
    print('   ' + sql_query.replace('\n', '\n   '))
    print()
    
    try:
        result_df = run_query(sql_query)
        print(f'✅ Result ({len(result_df)} rows):')
        print(result_df.to_string(index=False))
    except Exception as e:
        print(f'⚠️  Query error: {e}')
        result_df = None
    
    return sql_query, result_df

print('✅ NL→SQL engine ready. Use nl_to_sql("your question") to query.')

In [ ]:
# ── Example NL Queries ───────────────────────────────────────
# Run each cell separately to see results

print('='*60)
print('DEMO 1 — Revenue by state')
print('='*60)
nl_to_sql('Show me total revenue by customer state, top 5 only')

In [ ]:
print('='*60)
print('DEMO 2 — Monthly growth')
print('='*60)
nl_to_sql('What are the monthly order counts for 2018?')

In [ ]:
print('='*60)
print('DEMO 3 — Top sellers')
print('='*60)
nl_to_sql('Who are the top 5 sellers by total revenue?')

In [ ]:
print('='*60)
print('DEMO 4 — Payment behaviour')
print('='*60)
nl_to_sql('What percentage of orders use credit card vs other payment types?')

In [ ]:
# ── Interactive Query Cell ───────────────────────────────────
# Type YOUR OWN question here and run the cell

my_question = "Show average order value by product category, top 10"

print('='*60)
print('CUSTOM QUERY')
print('='*60)
nl_to_sql(my_question)

---
## 📋 MODULE 6 — GTM KPI Summary Dashboard (Text)
> *Final executive summary combining all KPIs — ready for Streamlit rendering*

In [ ]:
# ── Full GTM KPI Summary ─────────────────────────────────────
print()
print('╔' + '═'*58 + '╗')
print('║   GTM BI PLATFORM — EXECUTIVE DASHBOARD SUMMARY        ║')
print('╠' + '═'*58 + '╣')
print(f'║  💰 Revenue YoY Growth        :  {yoy_growth:>+.1f}%{" "*25}║')
print(f'║  📈 Avg MoM Growth            :  {avg_mom_growth:>+.1f}%{" "*25}║')
print(f'║  👥 Customer Retention Rate   :  {retention_rate:.2f}%{" "*24}║')
print(f'║  🔄 Customer Churn Rate       :  {churn_rate:.2f}%{" "*24}║')
print(f'║  🏆 Top Category              :  {df_cat.iloc[0]["category"][:28]:<28}║')
print(f'║  🌍 Top Region                :  {df_reg.iloc[0]["state"]:<28}║')
print('╚' + '═'*58 + '╝')
print()
print('🤖 LLM Insight Layer: ACTIVE (Claude API connected)')
print('💬 NL→SQL Engine: ACTIVE (ask any business question)')
print('📊 Streamlit Dashboard: run → streamlit run streamlit_app.py')